### RAG pipeline - data ingestion to vector db

In [3]:
# import required libraries
import os
from langchain_community.document_loaders import PyMuPDFLoader, PyPDFLoader, DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()
HF_TOKEN = os.getenv('HUGGING_FACE_API_KEY')

/tmp/ipykernel_231167/852626585.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyMuPDFLoader, PyPDFLoader, DirectoryLoader


In [19]:
# Read all the pdf inside diractory
def load_all_pdf(path_to_pdfs):
    """Load and process all the pdfs"""
    all_documents = []
    path_dir = Path(path_to_pdfs)

    # load all pdf as list
    files = list(path_dir.glob('**/*.pdf'))

    if not files:
        return None
    print(f"Found {len(files)} pdfs.")
    
    for file in files:
        print(f"prpcessing {file}")
        try:
            loader = PyMuPDFLoader(str(file))
            documents = loader.load()
            print(f'found {len(documents)} pages\n')

            for doc in documents:
                doc.metadata['source_file'] = file.name
                doc.metadata['file_type'] = 'pdf'

            all_documents.extend(documents)
        except Exception as e:
            print(f"Error: {e}")
    
    return all_documents


# less control over the files. Instead return all documents at once
def load_all_pdf2(dir):
    loader = DirectoryLoader(
        dir,
        glob='**/*.pdf',
        loader_cls=PyMuPDFLoader
    )
    all_documents = loader.load()

    for doc in all_documents:
        doc.metadata['source_file'] = Path(doc.metadata['file_path']).name
        doc.metadata['file_type'] = 'pdf'
    
    return all_documents

documents = load_all_pdf('../data/pdf/AstroPhysics/')
documents

Found 4 pdfs.
prpcessing ../data/pdf/AstroPhysics/ImpactofPerfectFluidDarkMatter.pdf
found 33 pages

prpcessing ../data/pdf/AstroPhysics/PropagatingInstabilityforWaveDarkMatter.pdf
found 33 pages

prpcessing ../data/pdf/AstroPhysics/ParticlePhysicsandModern.pdf
found 24 pages

prpcessing ../data/pdf/AstroPhysics/Core-HaloRelationofQuantumWaveDarkMatter.pdf
found 6 pages



[Document(metadata={'producer': 'pikepdf 8.15.1', 'creator': 'arXiv GenPDF (tex2pdf:57610bf)', 'creationdate': '', 'source': '../data/pdf/AstroPhysics/ImpactofPerfectFluidDarkMatter.pdf', 'file_path': '../data/pdf/AstroPhysics/ImpactofPerfectFluidDarkMatter.pdf', 'total_pages': 33, 'format': 'PDF 1.7', 'title': 'Impact of Perfect Fluid Dark Matter on the Appearance of Rotating Black Hole', 'author': 'Huang Yu-Xiang; Guo Sen; Liang En-Wei; Lin Kai', 'subject': '', 'keywords': '', 'moddate': '', 'trapped': '', 'modDate': '', 'creationDate': '', 'page': 0, 'source_file': 'ImpactofPerfectFluidDarkMatter.pdf', 'file_type': 'pdf'}, page_content='Impact of Perfect Fluid Dark Matter on the\nAppearance of Rotating Black Hole\nYu-Xiang Huanga, Sen Guob,∗, En-Wei Lianga,∗, Kai Linc\naGuangxi Key Laboratory for Relativistic Astrophysics, School of Physical Science and\nTechnology, Guangxi University, Nanning, 530004, People’s Republic of China\nbCollege of Physics and Electronic Engineering, Chong

In [3]:
# Split into chunks 
def split_text(docs, chunk_size = 1000, chunk_overlap = 200):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap,
        length_function = len,
        separators=['\n\n', '\n', '. ', '! ', '? ', ' ', '']
    )
    chunk = splitter.split_documents(documents=docs)
    print(f'Splitted {len(docs)} documents into {len(chunk)} chunk')

    if chunk:
        print(f'{chunk[0].metadata}')
        print(f'{chunk[0].page_content[:200]}')

    return chunk

chunks = split_text(documents)

Splitted 96 documents into 382 chunk
{'producer': 'pikepdf 8.15.1', 'creator': 'arXiv GenPDF (tex2pdf:57610bf)', 'creationdate': '', 'source': '../data/pdf/AstroPhysics/ImpactofPerfectFluidDarkMatter.pdf', 'file_path': '../data/pdf/AstroPhysics/ImpactofPerfectFluidDarkMatter.pdf', 'total_pages': 33, 'format': 'PDF 1.7', 'title': 'Impact of Perfect Fluid Dark Matter on the Appearance of Rotating Black Hole', 'author': 'Huang Yu-Xiang; Guo Sen; Liang En-Wei; Lin Kai', 'subject': '', 'keywords': '', 'moddate': '', 'trapped': '', 'modDate': '', 'creationDate': '', 'page': 0, 'source_file': 'ImpactofPerfectFluidDarkMatter.pdf', 'file_type': 'pdf'}
Impact of Perfect Fluid Dark Matter on the
Appearance of Rotating Black Hole
Yu-Xiang Huanga, Sen Guob,∗, En-Wei Lianga,∗, Kai Linc
aGuangxi Key Laboratory for Relativistic Astrophysics, School of Phy


In [4]:
models = [
    'sentence-transformers/all-MiniLM-L6-v2',
    'BAAI/bge-base-en-v1.5',    
    "BAAI/bge-small-en-v1.5",
    "intfloat/e5-base-v2"
]

In [5]:
## Embedding and vector store
from langchain_huggingface import HuggingFaceEndpointEmbeddings
from langchain_chroma import Chroma


hf_embeddings = HuggingFaceEndpointEmbeddings(  
    model="sentence-transformers/all-mpnet-base-v2",  
    task="feature-extraction",  
    huggingfacehub_api_token=HF_TOKEN,  
)

vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=hf_embeddings,
    collection_name='chromaDb',
    persist_directory='../chroma_db'
)

print("Vector store created successfully!")

/home/desktop-potato/Desktop/ai/prj/rag-app/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Vector store created successfully!


In [6]:
from typing import List

class RAGRetriever:
    """Handles query based retrival from vector-store"""

    def __init__(self, vector_store = vector_store, embeddings = hf_embeddings):
        """
        Initialize Retriever
        ARG:
            vector_store: vector store containing the embeddings
            embeddings: embedding function to handle query embedding
        """

        self.vector_store = vector_store
        self.embeddings = embeddings
    
    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[dict[str, any]]:
        """
        Retrieve relevent document for a querry 

        arg:
            query: The search query 
            top_k: Number of top best result to return 
            score_threshold: Minimum similarity score
        Returns:
            List of relevent document and metadata
        """

        print(f'Retrieving relevent data for querry "{query}"')
        print(f'Top_k: {top_k}, score_threshold: {score_threshold}')

        # Generating query embeddings
        query_embedding = self.embeddings.embed_query(query)

        # Search in vector db
        try:
            results = vector_store._collection.query(
                query_embeddings=query_embedding,
                n_results=top_k
            )

            # process result
            retrived_docs = []

            if results['documents'] and results['documents'][0]:
                doc_ids = results['ids'][0]
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]

                for i, (doc_id, document, metadata, distance) in enumerate(zip(doc_ids, documents, metadatas, distances)):
                    # Convert distance into similarity score
                    similarity_score = 1 - distance

                    if similarity_score >= score_threshold:
                        retrived_docs.append({
                            'doc_id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f'Retrive {len(retrived_docs)} documents after filtering.')
            else:
                print('No document found')
            
            return retrived_docs
                

        except Exception as e:
            print(f'Exception occured with {e}')

retriver = RAGRetriever()

In [7]:
#  An example how to use the retriver
result = retriver.retrieve("Dark matter")

Retrieving relevent data for querry "Dark matter"
Top_k: 5, score_threshold: 0.0
Retrive 5 documents after filtering.


In [4]:
groq_model = "llama-3.1-8b-instant"
GROQ_API_KEY = os.getenv('GROQ_API_KEY')

In [ ]:
from langchain_groq import ChatGroq
from langchain_classic.prompts import PromptTemplate
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage

# Initialize the groq llm
llm = ChatGroq(
    api_key=GROQ_API_KEY,
    model=groq_model,
    temperature=0.1,
    max_token = 1024
)

class GroqLLM: 
    def __init__(self, model_name: str, api_key: str, temperature: float = 0.1, max_token: int = 1024):
        """
        Initialize Groq LLM
        
        Args:
            model_name: Groq model name (qwen2-72b-instruct, llama3-70b-8192, etc.)
            api_key: Groq API key (or set GROQ_API_KEY environment variable)
        """

        if api_key is None:
            raise ValueError('groq api key required')
        self.model_name = model_name
        self.api_key = api_key

        # Initialize the groq llm
        self.llm = ChatGroq(
            api_key=GROQ_API_KEY,
            model=groq_model,
            temperature=temperature,
            max_token = max_token
        )
    
        print("Groq llm initialized")
    
    def generate_response(self, query: str, context: str, max_length: int = 500) :
        """
        Generate response using the retrieved context
        
        Args:
            query: User question
            context: Retrieved document context
            max_length: Maximum response length
            
        Returns:
            Generated response string
        """

        # Create prompt template
        prompt_template = PromptTemplate(
            input_variables=["context"],
            template="You are a helpful AI assistant. Use the following context to answer the question accurately and concisely.\n Context: {context}\nAnswer: Provide a clear and informative answer based on the context above. If the context doesn't contain enough information to answer the question, say so."
        )
        
        formatted_prompt = prompt_template(context=context, question=query)

        messages = [
            SystemMessage(content=formatted_prompt),
            HumanMessage(content=query)
        ]
        try:
            response = self.llm.invoke(messages)
            return response.content
        except Exception as e:
            return f"Exception : {e}"
        
groqLLM  = GroqLLM(model_name="llama-3.1-8b-instant", api_key=GROQ_API_KEY)

Groq llm initialized


/home/desktop-potato/Desktop/ai/prj/rag-app/.venv/lib/python3.12/site-packages/pydantic/main.py:263: UserWarning: WARNING! max_token is not default parameter.
                    max_token was transferred to model_kwargs.
                    Please confirm that max_token is what you intended.
  validated_self = self.__pydantic_validator__.validate_python(data, self_instance=self)


In [ ]:
# Simple rag function: Context ritrival + generating response
def rag_func(query: str, retriver: RAGRetriever, llm) :
    # Retrive the context from vector database
    results = retriver.retrieve(query=query, top_k=5, score_threshold=0.0)
    context = '\n\n'.join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relevent context found to answer the question"
    
    # Generate answer using llm
    ## generate the answwer using GROQ LLM
    prompt=f"""Use the following context to answer the question concisely.
        Context:
        {context}

        Question: {query}

        Answer:"""
    answer = llm.invoke(prompt.format(context = context, query = query))

generated_answer = rag_func(query='what is dark matter', retriver=retriver, llm=llm)

In [ ]:
# --- Advanced RAG Pipeline: Streaming, Citations, History, Summarization ---
class RAGPipeline:
    _MAX_MESSAGES = 20
    def __init__(self, llm, retriver):
        if llm is None or retriver is None:
            raise ValueError("Both llm and retriver is reauired")
        
        self.llm = llm
        self.retriver = retriver
        self.history = []

        print("Pipeline initialized.")
    
    def summarize_history(self):
        prompt = [
            SystemMessage(
                    content="""
                    Summarize the conversation.
                    Keep:
                    - User goals
                    - Important facts
                    - Decisions made
                    - Open questions

                    Be concise.
                    """
            )
        ] + self.history

        summary = self.llm.invoke(prompt).content

        self.history = self.history[:2] + [
            SystemMessage(content=f"Conversation summary:\n{summary}")
        ] + self.history[-2:]
    
    def query(self, query: str, top_k: int = 5, min_score: float = 0.2, stream: bool = False, summurize: bool = False) -> dict[str, any]:
        if len(self.history) > self.MAX_MESSAGES:
            self.summarize_history() 
        # Retrive related context
        retrived_context = retriver.retrieve(query=query, top_k=top_k, score_threshold=min_score)
        if not retrived_context:
            answer = "No relevent content found"
        else:
            context = "\n".join(doc['content'] for doc in retrived_context)
            sources = [{
                'source': doc.metadata['source_file'], 
                'page': doc.metadata['page'],
                'score': doc.metadata['similarity_score'] 
                } for doc in retrived_context
            ]
            promt_template = PromptTemplate(
                input_variables=[context, query],
                template='Answer the users QUESTION using the DOCUMENT text above.Keep your answer ground in the facts of the DOCUMENT.If the DOCUMENT doesnot contain the facts to answer the QUESTION return NONE, \n\n '
            )
            prompt = promt_template.format(context=context, qyery = query)
            self.history.append(HumanMessage(content=query))
            response = self.llm.invoke(prompt)
            self.history.append(AIMessage(content=response.content))
            answer = response.content

        return answer            

In [ ]:
l = [1,2,3,4,5,6,7,8]
l = l[:2] + [999] + l[-2:] 
l

[1, 2, 999, 7, 8]